# Analysis and Classification of Attacks using Realistic Botnet Dataset (Deep Learning)

本版本延續原始 notebook 的流程，但分類器改為深度學習（Multi-Task MLP）。

## Section 1: Import Libraries（分區意義）

這一區只做環境準備：匯入資料處理、模型訓練、評估需要的套件。

In [ ]:
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("default")

## Section 2: Configuration（分區意義）

集中管理路徑、特徵欄位、訓練參數，後續調整只需改這一區。

In [ ]:
RANDOM_STATE = 42

TRAIN_PATH = "UNSW_2018_IoT_Botnet_Final_10_best_Training.csv"
TEST_PATH = "UNSW_2018_IoT_Botnet_Final_10_best_Testing.csv"

FEATURE_COLUMNS = [
    "seq", "stddev", "N_IN_Conn_P_SrcIP", "min", "state_number",
    "mean", "N_IN_Conn_P_DstIP", "drate", "srate", "max"
]

TARGET_COLUMNS = ["attack", "category", "subcategory"]

VALID_SIZE = 0.25
BATCH_SIZE = 4096
EPOCHS = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

# CPU only machine can set True to reduce training time.
USE_SUBSET = True
SUBSET_SIZE = 600000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
def seed_everything(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_STATE)

## Section 3: Load Dataset & Quick EDA（分區意義）

讀取資料並快速檢查分布，確認不平衡情況與資料規模。

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

In [ ]:
for col in TARGET_COLUMNS:
    print(f"\n{col} distribution (train):")
    print(train_df[col].value_counts().head(10))

In [ ]:
if USE_SUBSET and SUBSET_SIZE < len(train_df):
    subset_ratio = SUBSET_SIZE / len(train_df)
    strat_key = train_df["attack"].astype(str) + "_" + train_df["category"].astype(str) + "_" + train_df["subcategory"].astype(str)
    train_df, _ = train_test_split(
        train_df,
        train_size=SUBSET_SIZE,
        random_state=RANDOM_STATE,
        stratify=strat_key
    )
    print(f"Subset enabled: {subset_ratio:.2%} of original data")
    print("New train shape:", train_df.shape)

## Section 4: Data Preprocessing（分區意義）

做標籤編碼、切分訓練/驗證集、標準化，讓模型可穩定學習。

In [ ]:
X_all = train_df[FEATURE_COLUMNS].copy()
y_all = train_df[TARGET_COLUMNS].copy()

X_test_official = test_df[FEATURE_COLUMNS].copy()
y_test_official = test_df[TARGET_COLUMNS].copy()

le_category = LabelEncoder()
le_subcategory = LabelEncoder()

y_all["category"] = le_category.fit_transform(y_all["category"])
y_all["subcategory"] = le_subcategory.fit_transform(y_all["subcategory"])

y_test_official["category"] = le_category.transform(y_test_official["category"])
y_test_official["subcategory"] = le_subcategory.transform(y_test_official["subcategory"])

In [ ]:
stratify_key = y_all["attack"].astype(str) + "_" + y_all["category"].astype(str) + "_" + y_all["subcategory"].astype(str)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_all,
    y_all,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_key
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_valid = scaler.transform(X_valid).astype(np.float32)

X_all_scaled = scaler.fit_transform(X_all).astype(np.float32)
X_test_official_scaled = scaler.transform(X_test_official).astype(np.float32)

print("X_train:", X_train.shape, "X_valid:", X_valid.shape)

In [ ]:
def build_loaders(X, y, batch_size=BATCH_SIZE, shuffle=True):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y["attack"].to_numpy(), dtype=torch.float32).view(-1, 1),
        torch.tensor(y["category"].to_numpy(), dtype=torch.long),
        torch.tensor(y["subcategory"].to_numpy(), dtype=torch.long),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = build_loaders(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = build_loaders(X_valid, y_valid, batch_size=BATCH_SIZE * 2, shuffle=False)

In [ ]:
# Class imbalance weights
attack_values = y_train["attack"].to_numpy()
pos_count = attack_values.sum()
neg_count = len(attack_values) - pos_count
pos_weight = torch.tensor([neg_count / max(pos_count, 1.0)], dtype=torch.float32, device=DEVICE)

cat_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train["category"]),
    y=y_train["category"].to_numpy()
)
sub_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train["subcategory"]),
    y=y_train["subcategory"].to_numpy()
)

cat_weights = torch.tensor(cat_weights, dtype=torch.float32, device=DEVICE)
sub_weights = torch.tensor(sub_weights, dtype=torch.float32, device=DEVICE)

## Section 5: Build Deep Learning Model（分區意義）

建立多任務 MLP：共享特徵主幹，再分三個輸出頭同時預測三個標籤。

In [ ]:
class MultiTaskMLP(nn.Module):
    def __init__(self, input_dim, n_category, n_subcategory):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
        )
        self.attack_head = nn.Linear(64, 1)
        self.category_head = nn.Linear(64, n_category)
        self.subcategory_head = nn.Linear(64, n_subcategory)

    def forward(self, x):
        h = self.backbone(x)
        out_attack = self.attack_head(h)
        out_category = self.category_head(h)
        out_subcategory = self.subcategory_head(h)
        return out_attack, out_category, out_subcategory

model = MultiTaskMLP(
    input_dim=len(FEATURE_COLUMNS),
    n_category=len(le_category.classes_),
    n_subcategory=len(le_subcategory.classes_),
).to(DEVICE)

criterion_attack = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion_category = nn.CrossEntropyLoss(weight=cat_weights)
criterion_subcategory = nn.CrossEntropyLoss(weight=sub_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

## Section 6: Train & Validate（分區意義）

訓練模型並監控驗證集分數，確認是否學到可泛化的分類能力。

In [ ]:
def evaluate_loader(model, loader, device=DEVICE):
    model.eval()

    all_attack_true, all_attack_pred = [], []
    all_cat_true, all_cat_pred = [], []
    all_sub_true, all_sub_pred = [], []

    with torch.no_grad():
        for xb, ya, yc, ys in loader:
            xb = xb.to(device)
            ya = ya.to(device)
            yc = yc.to(device)
            ys = ys.to(device)

            out_a, out_c, out_s = model(xb)

            pred_a = (torch.sigmoid(out_a) >= 0.5).long().view(-1).cpu().numpy()
            pred_c = torch.argmax(out_c, dim=1).cpu().numpy()
            pred_s = torch.argmax(out_s, dim=1).cpu().numpy()

            all_attack_true.extend(ya.view(-1).long().cpu().numpy())
            all_attack_pred.extend(pred_a)
            all_cat_true.extend(yc.cpu().numpy())
            all_cat_pred.extend(pred_c)
            all_sub_true.extend(ys.cpu().numpy())
            all_sub_pred.extend(pred_s)

    result = {
        "attack": {
            "accuracy": accuracy_score(all_attack_true, all_attack_pred),
            "balanced_accuracy": balanced_accuracy_score(all_attack_true, all_attack_pred),
            "macro_f1": f1_score(all_attack_true, all_attack_pred, average="macro", zero_division=0),
            "report": classification_report(all_attack_true, all_attack_pred, zero_division=0),
        },
        "category": {
            "accuracy": accuracy_score(all_cat_true, all_cat_pred),
            "balanced_accuracy": balanced_accuracy_score(all_cat_true, all_cat_pred),
            "macro_f1": f1_score(all_cat_true, all_cat_pred, average="macro", zero_division=0),
            "report": classification_report(all_cat_true, all_cat_pred, zero_division=0),
        },
        "subcategory": {
            "accuracy": accuracy_score(all_sub_true, all_sub_pred),
            "balanced_accuracy": balanced_accuracy_score(all_sub_true, all_sub_pred),
            "macro_f1": f1_score(all_sub_true, all_sub_pred, average="macro", zero_division=0),
            "report": classification_report(all_sub_true, all_sub_pred, zero_division=0),
        },
    }

    return result

best_score = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for xb, ya, yc, ys in train_loader:
        xb = xb.to(DEVICE)
        ya = ya.to(DEVICE)
        yc = yc.to(DEVICE)
        ys = ys.to(DEVICE)

        out_a, out_c, out_s = model(xb)

        loss_a = criterion_attack(out_a, ya)
        loss_c = criterion_category(out_c, yc)
        loss_s = criterion_subcategory(out_s, ys)

        loss = loss_a + loss_c + loss_s

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    metrics = evaluate_loader(model, valid_loader, device=DEVICE)
    score = (metrics["attack"]["macro_f1"] + metrics["category"]["macro_f1"] + metrics["subcategory"]["macro_f1"]) / 3.0

    if score > best_score:
        best_score = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch:02d} | train_loss={running_loss/len(train_loader):.4f} | "
        f"valid_macro_f1: attack={metrics['attack']['macro_f1']:.4f}, "
        f"category={metrics['category']['macro_f1']:.4f}, "
        f"subcategory={metrics['subcategory']['macro_f1']:.4f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)
    print("Best validation macro_f1(avg):", round(best_score, 4))

In [ ]:
valid_metrics = evaluate_loader(model, valid_loader, device=DEVICE)

for task_name in ["attack", "category", "subcategory"]:
    print(f"\nValidation Report: {task_name}")
    print(valid_metrics[task_name]["report"])

## Section 7: Train on Full Train Set and Evaluate on Official Test（分區意義）

使用完整訓練資料重訓，最後在官方 testing 檔案評估，和原 notebook 的最終流程一致。

In [ ]:
full_loader = build_loaders(X_all_scaled, y_all, batch_size=BATCH_SIZE, shuffle=True)
test_loader = build_loaders(X_test_official_scaled, y_test_official, batch_size=BATCH_SIZE * 2, shuffle=False)

final_model = MultiTaskMLP(
    input_dim=len(FEATURE_COLUMNS),
    n_category=len(le_category.classes_),
    n_subcategory=len(le_subcategory.classes_),
).to(DEVICE)

final_optimizer = torch.optim.Adam(final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

for epoch in range(1, max(4, EPOCHS // 2) + 1):
    final_model.train()
    run_loss = 0.0

    for xb, ya, yc, ys in full_loader:
        xb = xb.to(DEVICE)
        ya = ya.to(DEVICE)
        yc = yc.to(DEVICE)
        ys = ys.to(DEVICE)

        out_a, out_c, out_s = final_model(xb)
        loss = criterion_attack(out_a, ya) + criterion_category(out_c, yc) + criterion_subcategory(out_s, ys)

        final_optimizer.zero_grad()
        loss.backward()
        final_optimizer.step()

        run_loss += loss.item()

    print(f"Full-train epoch {epoch:02d} | loss={run_loss/len(full_loader):.4f}")

In [ ]:
test_metrics = evaluate_loader(final_model, test_loader, device=DEVICE)

for task_name in ["attack", "category", "subcategory"]:
    print(f"\nOfficial Test Report: {task_name}")
    print(test_metrics[task_name]["report"])

## End of Notebook

你可以先用 `USE_SUBSET=True` 快速驗證流程，再改成 `False` 跑完整資料。